# Eflux2 applied to *A. niger*

In [42]:
import sys
import pandas as pd
import cobra
sys.path.append('../../src')
from eflux2 import EFlux2

## Read global proteomics normalized to the reference strain SF ABF180_17_R2

[(βAl-3HP)++, pyc2++, Δald6a+NeoR] (Strain ABF-10216R8, ICE ID ABF_011245)

In [43]:
global_prot = pd.read_csv('../../data/round2/normalized_global_proteomics.csv', index_col='Transcript')
global_prot

,SF ABF180_1_R1,SF ABF180_1_R2,SF ABF180_1_R3,SF ABF180_2_R1,SF ABF180_2_R2,SF ABF180_2_R3,SF ABF180_3_R1,SF ABF180_3_R2,SF ABF180_3_R3,SF ABF180_4_R1,...,SF ABF180_21_R3,SF ABF180_22_R1,SF ABF180_22_R2,SF ABF180_22_R3,SF ABF180_23_R1,SF ABF180_23_R2,SF ABF180_23_R3,SF ABF180_24_R1,SF ABF180_24_R2,SF ABF180_24_R3
Transcript,,,,,,,,,,,,,,,,,,,,,
1170085,0.967266,0.996633,0.992893,0.988900,1.005546,0.994246,0.995436,0.984235,0.985065,0.987202,...,0.996191,1.000241,0.981752,1.003174,1.000581,1.006389,0.998983,0.974427,0.986005,0.982189
1141495,1.012991,1.010041,1.011290,1.012467,1.008045,1.010177,1.032464,1.009299,1.012114,1.014553,...,1.010923,1.019918,1.003567,1.020978,1.010532,1.022859,1.026123,1.015454,1.016219,1.003530
201546,0.979618,0.980405,0.973220,0.981140,0.998126,0.998079,0.991908,0.981119,0.990187,0.990484,...,0.988765,0.994606,0.979760,0.993260,1.001902,0.996684,0.994010,0.982122,0.986781,0.980817
1147898,0.995988,1.000736,0.999673,1.021090,1.020466,1.023879,1.007351,1.005843,1.003600,1.006999,...,1.007180,1.001447,1.008006,1.004574,1.003261,1.014986,1.022990,1.002505,0.996382,1.002931
1142869,1.018378,1.017542,1.017273,1.015201,1.009343,1.003975,1.016892,1.010228,1.026211,1.010030,...,1.000423,1.002124,1.005939,1.004112,1.001554,1.010091,0.992440,1.000478,0.998578,0.997856
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1117615,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
210213,1.000000,1.000000,1.000000,1.000000,1.000000,0.926492,0.928163,1.000000,0.951025,1.000000,...,1.013350,1.024441,1.026118,1.015304,1.012428,0.997734,0.990888,0.970595,1.000000,0.954842
1119048,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Read iJB1325 with non-native genes added for 3HP production

In [44]:
# todo: update gurobi license (not required to use gurobi)
model = cobra.io.load_json_model('../../models/iJB1325_HP.json')
#model.solver = 'gurobi'
#model.solver

In [45]:
model.medium

{'BOUNDARY_GLCe': 10.0,
 'BOUNDARY_H2Oe': 1000.0,
 'BOUNDARY_HNO3e': 1000.0,
 'BOUNDARY_O2e': 1000.0,
 'BOUNDARY_PIe': 1000.0,
 'BOUNDARY_SLFe': 1000.0,
 'BOUNDARY_Ke': 1000.0,
 'BOUNDARY_CAe': 1000.0}

In [46]:
model.reactions.BOUNDARY_HNO3e.lower_bound = 0.0
model.reactions.BOUNDARY_NH4e.lower_bound = -1000.0

## Read line rates

In [48]:
line_rates = pd.read_csv('../../data/round2/Line_rates.csv', index_col='Line Name')
line_rates.head()

,Omics Sample ID,Strain (ICE),Genotype,GLCe (mmol/gDW/hr),3hpp_e (mmol/gDW/hr),ETHe (mmol/gDW/hr),EOLe (mmol/gDW/hr),CITe (mmol/gDW/hr)
Line Name,,,,,,,,
SF ABF180_1_R1,SF ABF180_1_R1,ABF_008340,wild-type,-3.824936,0.000000,-0.792609,0.003407,0.013740
SF ABF180_1_R2,SF ABF180_1_R2,ABF_008340,wild-type,-3.557249,0.000000,-0.678005,0.012109,0.009979
SF ABF180_1_R3,SF ABF180_1_R3,ABF_008340,wild-type,-3.415471,0.000000,-1.080524,0.010234,0.009554
SF ABF180_2_R1,SF ABF180_2_R1,ABF_008343,(βAl-3HP)+,-5.928543,2.040397,-0.352736,0.022628,-0.001312
SF ABF180_2_R2,SF ABF180_2_R2,ABF_008343,(βAl-3HP)+,-5.792077,2.099299,-0.539260,0.005308,-0.002935


In [49]:
# Drop ethanol uptake - this is likely evaporation
line_rates = line_rates.drop(columns='ETHe (mmol/gDW/hr)')

## Normalize uptake and secretion rates by glucose uptake rate of reference study line.

In [50]:
ref_line = 'SF ABF180_17_R2'

ref_line_glucose_rate = -line_rates.loc[ref_line,'GLCe (mmol/gDW/hr)']
ref_line_glucose_rate

6.277127634711532

In [51]:
normalized_line_rates = line_rates.drop(['Omics Sample ID', 'Strain (ICE)', 'Genotype'], axis=1)
normalized_line_rates = normalized_line_rates.divide(ref_line_glucose_rate, axis=1)
normalized_line_rates

,GLCe (mmol/gDW/hr),3hpp_e (mmol/gDW/hr),EOLe (mmol/gDW/hr),CITe (mmol/gDW/hr)
Line Name,,,,
SF ABF180_1_R1,-0.609345,0.000000,0.000543,0.002189
SF ABF180_1_R2,-0.566700,0.000000,0.001929,0.001590
SF ABF180_1_R3,-0.544114,0.000000,0.001630,0.001522
SF ABF180_2_R1,-0.944467,0.325053,0.003605,-0.000209
SF ABF180_2_R2,-0.922727,0.334436,0.000846,-0.000467
...,...,...,...,...
SF ABF180_23_R2,-0.989133,0.478907,0.012959,0.001360
SF ABF180_23_R3,-1.137562,0.564224,0.013712,0.001255
SF ABF180_24_R1,-1.124896,0.949444,0.024140,0.002688


In [52]:
normalized_line_rates[(normalized_line_rates < 0).any(axis=1)]

,GLCe (mmol/gDW/hr),3hpp_e (mmol/gDW/hr),EOLe (mmol/gDW/hr),CITe (mmol/gDW/hr)
Line Name,,,,
SF ABF180_1_R1,-0.609345,0.000000,0.000543,0.002189
SF ABF180_1_R2,-0.566700,0.000000,0.001929,0.001590
SF ABF180_1_R3,-0.544114,0.000000,0.001630,0.001522
SF ABF180_2_R1,-0.944467,0.325053,0.003605,-0.000209
SF ABF180_2_R2,-0.922727,0.334436,0.000846,-0.000467
...,...,...,...,...
SF ABF180_23_R2,-0.989133,0.478907,0.012959,0.001360
SF ABF180_23_R3,-1.137562,0.564224,0.013712,0.001255
SF ABF180_24_R1,-1.124896,0.949444,0.024140,0.002688


In [53]:
(normalized_line_rates < 0).any()

GLCe (mmol/gDW/hr)       True
3hpp_e (mmol/gDW/hr)    False
EOLe (mmol/gDW/hr)      False
CITe (mmol/gDW/hr)       True
dtype: bool

In [12]:
# Instead of replacing negative values with zero
# normalized_line_rates[normalized_line_rates < 0] = 0.0
# Use negative values for uptake and positive values for secretion

In [9]:
normalized_line_rates.to_csv('../../data/round2/normalized_line_rates.csv')

### Reduce the model based on the FBA solution maximizing CO2 production in the reference strain 

In [55]:
const_dict = {
       'BOUNDARY_GLCe': 'GLCe (mmol/gDW/hr)',
       'EX_3hpp_e': '3hpp_e (mmol/gDW/hr)',
       #'BOUNDARY_ETHe': 'ETHe (mmol/gDW/hr)',
       'BOUNDARY_EOLe': 'EOLe (mmol/gDW/hr)',
       'BOUNDARY_CITe': 'CITe (mmol/gDW/hr)'
}

In [56]:
with model:
    for k, v in const_dict.items():
        model.reactions.get_by_id(k).lower_bound = normalized_line_rates[v][ref_line]
    model.objective = 'BOUNDARY_CO2e'
    pfba_sol_ref = cobra.flux_analysis.pfba(model)

In [62]:
len(pfba_sol_ref[abs(pfba_sol_ref.fluxes) > 1e-9])

93

In [64]:
zero_flux_ref = pfba_sol_ref[abs(pfba_sol_ref.fluxes) < 1e-9].index
model.remove_reactions(zero_flux_ref, remove_orphans=True)

In [65]:
model

Name,iJB1325_ATCC1015
Memory address,7fee521e9438
Number of metabolites,138
Number of reactions,93
Number of genes,146
Number of groups,0
Objective expression,1.0*BOUNDARY_CO2e - 1.0*BOUNDARY_CO2e_reverse_2cdc1
Compartments,"Cytoplasm, Mitochondria, Extracellular"


## Run Eflux

In [66]:
fluxes = {}
for rep in global_prot.columns:
    with model:
        for k, v in const_dict.items():
            model.reactions.get_by_id(k).lower_bound = normalized_line_rates[v][rep]
        try:
            print(rep)
            fluxes[rep] = EFlux2(model, global_prot[rep])
        except TypeError:
            print(f"Replicate {rep} with glucose {glucose_uptake} and 3hp {secrete_3hp} is infeasible")

SF ABF180_1_R1
FBA status optimal
FBA solution 1.86287951370712
EFlux2 status optimal
EFlux2 solution 33.80125399895117
SF ABF180_1_R2
FBA status optimal
FBA solution 1.7308518612251667
EFlux2 status optimal
EFlux2 solution 29.15558791909489
SF ABF180_1_R3
FBA status optimal
FBA solution 1.662186123575238
EFlux2 status optimal
EFlux2 solution 26.891289857622816
SF ABF180_2_R1
FBA status optimal
FBA solution 2.288427035378078
EFlux2 status optimal
EFlux2 solution 59.37992661009217
SF ABF180_2_R2
FBA status optimal
FBA solution 2.2764924626531466
EFlux2 status optimal
EFlux2 solution 58.87997712102427
SF ABF180_2_R3
FBA status optimal
FBA solution 2.2735496876303514
EFlux2 status optimal
EFlux2 solution 58.78485731303181
SF ABF180_3_R1
FBA status optimal
FBA solution 2.2700681777641103
EFlux2 status optimal
EFlux2 solution 58.391931314291966
SF ABF180_3_R2
FBA status optimal
FBA solution 2.2792580739100132
EFlux2 status optimal
EFlux2 solution 58.721353168966225
SF ABF180_3_R3
FBA status

/Users/kimj972/anaconda3/envs/python3_cobrapy/lib/python3.6/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


EFlux2 status infeasible
EFlux2 solution 0.08721463752713277
SF ABF180_5_R2
FBA status infeasible
FBA solution 0.042408993134607144
EFlux2 status infeasible
EFlux2 solution 0.11638458967067322
SF ABF180_5_R3
FBA status infeasible
FBA solution 0.19086847792502332
EFlux2 status infeasible
EFlux2 solution 0.22181524514734008
SF ABF180_6_R1
FBA status optimal
FBA solution 2.101029843405702
EFlux2 status optimal
EFlux2 solution 45.88381911757923
SF ABF180_6_R2
FBA status optimal
FBA solution 2.2743700077984004
EFlux2 status optimal
EFlux2 solution 58.339048264111355
SF ABF180_6_R3
FBA status optimal
FBA solution 2.2644271793038455
EFlux2 status optimal
EFlux2 solution 58.02961252173907
SF ABF180_7_R1
FBA status optimal
FBA solution 2.213252360431179
EFlux2 status optimal
EFlux2 solution 52.83988667430047
SF ABF180_7_R2
FBA status optimal
FBA solution 2.265270856391254
EFlux2 status optimal
EFlux2 solution 58.14659228018724
SF ABF180_7_R3
FBA status optimal
FBA solution 2.2694028736714
EFlux

In [67]:
for rep in global_prot.columns:
    if fluxes[rep].status is not 'optimal':
        print(rep, fluxes[rep].status)

SF ABF180_5_R1 infeasible
SF ABF180_5_R2 infeasible
SF ABF180_5_R3 infeasible
SF ABF180_17_R1 infeasible
SF ABF180_17_R2 infeasible
SF ABF180_17_R3 infeasible
SF ABF180_18_R1 infeasible
SF ABF180_19_R1 infeasible
SF ABF180_19_R3 infeasible
SF ABF180_20_R1 infeasible
SF ABF180_20_R2 infeasible
SF ABF180_20_R3 infeasible
SF ABF180_21_R1 infeasible
SF ABF180_21_R2 infeasible
SF ABF180_21_R3 infeasible
SF ABF180_22_R1 infeasible
SF ABF180_22_R2 infeasible
SF ABF180_22_R3 infeasible


### get eflux2 solutions by adding slack variables for infeasible cases

In [23]:
import numpy as np
from optlang.symbolics import add
from eflux2 import get_eflux2_bounds
from eflux.eflux2 import add_slack_variables_to_model

In [28]:
for rep in global_prot.columns:
    if fluxes[rep].status is not 'optimal':
        print(rep)
        # get eflux2 bounds from global proteomics
        eflux2_bounds = get_eflux2_bounds(model, global_prot[rep])
        upper_bounds = {k: v[1] if v[1] <= 1000.0 else 1000.0 for k, v in eflux2_bounds.items()}
        # build a relaxed model with slack variables
        relaxed_model = add_slack_variables_to_model(model, upper_bounds)
        # solve FBA to calculate the maximum CO2
        relaxed_model.tolerance = 1e-9
        fba_sol = relaxed_model.optimize()
        print('FBA status', fba_sol.status)
        print('FBA solution', fba_sol.objective_value)
        # Constrain the growth to the optimal value
        for r in relaxed_model.reactions:
            if r.objective_coefficient == 1.0:
                r.lower_bound = fba_sol[r.id]
        # minimize the sum of squared flux values
        relaxed_model.objective = relaxed_model.problem.Objective(add([r.flux_expression**2 for r in relaxed_model.reactions]), direction='min')
        eflux2_sol = relaxed_model.optimize()
        print('EFlux2 status', eflux2_sol.status)
        print('EFlux2 solution', eflux2_sol.objective_value)
        fluxes[rep] = eflux2_sol

SF ABF180_5_R1
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution 4.9093324742347827e-11
SF ABF180_5_R2
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution 4.736805118201147e-11
SF ABF180_5_R3
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution -1.1967845666940655e-09
SF ABF180_17_R1
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution -2.5123421961024682e-11
SF ABF180_17_R2
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution -1.0824451831230847e-09
SF ABF180_17_R3
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution -2.133435419890147e-09
SF ABF180_19_R1
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution -1.2644389707123762e-10
SF ABF180_19_R3
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution -5.020976105324271e-10
SF ABF180_20_R1
FBA status optimal
FBA solution 0.0
EFlux2 status optimal
EFlux2 solution -1.737

In [138]:
for rep in global_prot.columns:
    if fluxes[rep].status is not 'optimal':
        print(rep, fluxes[rep].status)

In [142]:
for rep in global_prot.columns:
    print(rep, round(fluxes[rep].objective_value,3))

SF ABF180_1_R1 14.87
SF ABF180_1_R2 14.13
SF ABF180_1_R3 13.752
SF ABF180_2_R1 29.07
SF ABF180_2_R2 25.978
SF ABF180_2_R3 34.923
SF ABF180_3_R1 24.071
SF ABF180_3_R2 27.096
SF ABF180_3_R3 19.801
SF ABF180_4_R1 38.638
SF ABF180_4_R2 24.266
SF ABF180_4_R3 25.665
SF ABF180_5_R1 0.0
SF ABF180_5_R2 0.0
SF ABF180_5_R3 -0.0
SF ABF180_6_R1 14.944
SF ABF180_6_R2 18.595
SF ABF180_6_R3 18.648
SF ABF180_7_R1 18.702
SF ABF180_7_R2 26.328
SF ABF180_7_R3 22.683
SF ABF180_8_R1 20.314
SF ABF180_8_R2 16.595
SF ABF180_8_R3 25.375
SF ABF180_9_R1 32.711
SF ABF180_9_R2 18.063
SF ABF180_9_R3 29.124
SF ABF180_10_R1 20.818
SF ABF180_10_R2 36.134
SF ABF180_10_R3 13.093
SF ABF180_11_R1 29.995
SF ABF180_11_R2 52.29
SF ABF180_11_R3 39.671
SF ABF180_12_R1 51.043
SF ABF180_12_R2 63.796
SF ABF180_12_R3 48.125
SF ABF180_13_R1 31.678
SF ABF180_13_R2 35.07
SF ABF180_13_R3 27.434
SF ABF180_14_R1 34.143
SF ABF180_14_R2 29.919
SF ABF180_14_R3 36.618
SF ABF180_15_R1 31.83
SF ABF180_15_R2 25.637
SF ABF180_15_R3 25.364
SF ABF

### Adding slack variables resulted in zero growth and all-zero fluxes
Adding slack variables to the infeasible subset introduces bias in flux prediction
### Switching to increasing the arbitrary scale for normalized global proteomics

In [68]:
# Find the minimum scaling factor
scaling_factor = 1.0
scaling_finished = False
while not scaling_finished:
    print('scaling_factor:', round(scaling_factor,1))
    scaling_finished = True
    for rep in global_prot.columns:
        if fluxes[rep].status is not 'optimal':
            with model:
                for k, v in const_dict.items():
                    model.reactions.get_by_id(k).lower_bound = normalized_line_rates[v][rep]
                eflux2_bounds = get_eflux2_bounds(model, scaling_factor*global_prot[rep])
                for r in model.reactions:
                    r.bounds = eflux2_bounds[r.id]
                sol = model.optimize()
            print(rep, sol.status, sol.objective_value)
            if sol.status is not 'optimal':
                scaling_factor = scaling_factor + 0.1
                scaling_finished = False
                break
    if scaling_finished:
        print('Done')

scaling_factor: 1.0
SF ABF180_5_R1 infeasible 0.3938817906423824
scaling_factor: 1.1
SF ABF180_5_R1 optimal 2.0200603402074044
SF ABF180_5_R2 optimal 2.4917349909238995
SF ABF180_5_R3 infeasible 0.09162644321819989
scaling_factor: 1.2
SF ABF180_5_R1 optimal 2.7351499012325657
SF ABF180_5_R2 optimal 2.718332096206569
SF ABF180_5_R3 optimal 1.954872291422312
SF ABF180_17_R1 optimal 1.9915037170755006
SF ABF180_17_R2 infeasible 0.19774159602501357
scaling_factor: 1.3
SF ABF180_5_R1 optimal 2.963131064907479
SF ABF180_5_R2 optimal 2.8952887234443914
SF ABF180_5_R3 optimal 2.966833838711774
SF ABF180_17_R1 optimal 1.9915037170755006
SF ABF180_17_R2 optimal 2.1954785455652512
SF ABF180_17_R3 optimal 2.975584320579598
SF ABF180_18_R1 optimal 2.923650686332399
SF ABF180_19_R1 optimal 2.971120656124461
SF ABF180_19_R3 optimal 2.604010639333219
SF ABF180_20_R1 optimal 2.6676393390860946
SF ABF180_20_R2 optimal 2.983070865664022
SF ABF180_20_R3 optimal 2.9798418455564324
SF ABF180_21_R1 optimal 2

In [69]:
fluxes = {}
scaling_factor = 1.5
for rep in global_prot.columns:
    with model:
        for k, v in const_dict.items():
            model.reactions.get_by_id(k).lower_bound = normalized_line_rates[v][rep]
        try:
            print(rep)
            fluxes[rep] = EFlux2(model, scaling_factor*global_prot[rep])
        except TypeError:
            print(f"Replicate {rep} with glucose {glucose_uptake} and 3hp {secrete_3hp} is infeasible")

SF ABF180_1_R1
FBA status optimal
FBA solution 1.8628795137071203
EFlux2 status optimal
EFlux2 solution 33.80125399895118
SF ABF180_1_R2
FBA status optimal
FBA solution 1.7308518612251669
EFlux2 status optimal
EFlux2 solution 29.1555879190949
SF ABF180_1_R3
FBA status optimal
FBA solution 1.6621861235752382
EFlux2 status optimal
EFlux2 solution 26.891289857622816
SF ABF180_2_R1
FBA status optimal
FBA solution 2.886304110473323
EFlux2 status optimal
EFlux2 solution 81.03362362996859
SF ABF180_2_R2
FBA status optimal
FBA solution 2.8236713462076946
EFlux2 status optimal
EFlux2 solution 77.61952771291868
SF ABF180_2_R3
FBA status optimal
FBA solution 3.098216888489758
EFlux2 status optimal
EFlux2 solution 93.39989996289569
SF ABF180_3_R1
FBA status optimal
FBA solution 2.932254121890804
EFlux2 status optimal
EFlux2 solution 83.63503146717945
SF ABF180_3_R2
FBA status optimal
FBA solution 2.98552737809891
EFlux2 status optimal
EFlux2 solution 86.68033734861231
SF ABF180_3_R3
FBA status opt

FBA status optimal
FBA solution 3.0086128821130638
EFlux2 status optimal
EFlux2 solution 87.82890936373508
SF ABF180_23_R3
FBA status optimal
FBA solution 3.2539768009047583
EFlux2 status optimal
EFlux2 solution 108.75154102732135
SF ABF180_24_R1
FBA status optimal
FBA solution 3.236020821638173
EFlux2 status optimal
EFlux2 solution 106.18375082615182
SF ABF180_24_R2
FBA status optimal
FBA solution 2.4425435977161407
EFlux2 status optimal
EFlux2 solution 57.493308750321276
SF ABF180_24_R3
FBA status optimal
FBA solution 3.1543803086731343
EFlux2 status optimal
EFlux2 solution 96.38298277293649


In [70]:
for rep in global_prot.columns:
    if fluxes[rep].status is not 'optimal':
        print(rep, fluxes[rep].status)

In [72]:
eflux_rates = pd.DataFrame(dict([(rep, fluxes[rep].to_frame()['fluxes']) for rep in fluxes]))
eflux_rates.head()

,SF ABF180_1_R1,SF ABF180_1_R2,SF ABF180_1_R3,SF ABF180_2_R1,SF ABF180_2_R2,SF ABF180_2_R3,SF ABF180_3_R1,SF ABF180_3_R2,SF ABF180_3_R3,SF ABF180_4_R1,...,SF ABF180_21_R3,SF ABF180_22_R1,SF ABF180_22_R2,SF ABF180_22_R3,SF ABF180_23_R1,SF ABF180_23_R2,SF ABF180_23_R3,SF ABF180_24_R1,SF ABF180_24_R2,SF ABF180_24_R3
r5a,0.609345,0.566700,0.544114,0.944467,0.922727,1.013443,0.961490,0.979892,0.905427,1.199513,...,1.119526,1.257292,1.290709,1.252386,1.044309,0.989133,1.137562,1.124896,0.812758,1.040476
r10,0.306548,0.284844,0.273534,0.474772,0.464352,0.509618,0.482778,0.491707,0.453913,0.628005,...,0.586642,0.661700,0.681823,0.670494,0.523663,0.495416,0.586476,0.575134,0.403212,0.519815
r12a,0.306548,0.284844,0.273534,0.474772,0.464352,0.509618,0.482778,0.491707,0.453913,0.628005,...,0.586642,0.661700,0.681823,0.670494,0.523663,0.495416,0.586476,0.575134,0.403212,0.519815
r13a,0.306548,0.284844,0.273534,0.474772,0.464352,0.509618,0.482778,0.491707,0.453913,0.628005,...,0.586642,0.661700,0.681823,0.670494,0.523663,0.495416,0.586476,0.575134,0.403212,0.519815
r14,0.616847,0.572676,0.550021,0.954621,0.934682,1.025027,0.969622,0.986936,0.910224,1.312508,...,1.227040,1.389506,1.436581,1.429589,1.050341,0.992532,1.208343,1.175638,0.800091,1.038784


In [54]:
#eflux_rates.to_csv('../../data/round2/Eflux2_flux_rates_raw.csv')

In [73]:
eflux_rates = eflux_rates*ref_line_glucose_rate
eflux_rates.head()

,SF ABF180_1_R1,SF ABF180_1_R2,SF ABF180_1_R3,SF ABF180_2_R1,SF ABF180_2_R2,SF ABF180_2_R3,SF ABF180_3_R1,SF ABF180_3_R2,SF ABF180_3_R3,SF ABF180_4_R1,...,SF ABF180_21_R3,SF ABF180_22_R1,SF ABF180_22_R2,SF ABF180_22_R3,SF ABF180_23_R1,SF ABF180_23_R2,SF ABF180_23_R3,SF ABF180_24_R1,SF ABF180_24_R2,SF ABF180_24_R3
r5a,3.824936,3.557249,3.415471,5.928543,5.792077,6.361510,6.035393,6.150907,5.683483,7.529497,...,7.027410,7.892183,8.101947,7.861385,6.555260,6.208917,7.140623,7.061118,5.101785,6.531200
r10,1.924241,1.788003,1.717006,2.980206,2.914799,3.198934,3.030459,3.086508,2.849269,3.942069,...,3.682424,4.153573,4.279888,4.208774,3.287097,3.109792,3.681386,3.610186,2.531015,3.262944
r12a,1.924241,1.788003,1.717006,2.980206,2.914799,3.198934,3.030459,3.086508,2.849269,3.942069,...,3.682424,4.153573,4.279888,4.208774,3.287097,3.109792,3.681386,3.610186,2.531015,3.262944
r13a,1.924241,1.788003,1.717006,2.980206,2.914799,3.198934,3.030459,3.086508,2.849269,3.942069,...,3.682424,4.153573,4.279888,4.208774,3.287097,3.109792,3.681386,3.610186,2.531015,3.262944
r14,3.872027,3.594763,3.452554,5.992279,5.867121,6.434227,6.086442,6.195123,5.713593,8.238780,...,7.702288,8.722107,9.017603,8.973711,6.593127,6.230251,7.584920,7.379628,5.022274,6.520578


## Process eflux_rates and reduce metabolic model

In [80]:
eflux_rates[abs(eflux_rates) < 1e-8] = 0.0

In [81]:
print('number of reactions', len(eflux_rates))
print('all zero flux', len(eflux_rates[(abs(eflux_rates) == 0).all(axis=1)]))
print('non-zero flux', len(eflux_rates[(abs(eflux_rates) != 0).any(axis=1)]))

number of reactions 93
all zero flux 0
non-zero flux 93


In [82]:
# remove reactions with all zero flux
reduced_model = model.copy()
zero_flux = eflux_rates.index[(abs(eflux_rates) == 0).all(axis=1)]
reduced_model.remove_reactions(zero_flux, remove_orphans=True)
eflux_rates.drop(zero_flux, inplace=True)

In [83]:
print('non-zero', len(eflux_rates))
print('non-negative', len(eflux_rates[(eflux_rates >= 0).all(axis=1)]))
print('non-positive', len(eflux_rates[(eflux_rates <= 0).all(axis=1)]))
print('mixed sign', len(eflux_rates[(eflux_rates > 0).any(axis=1) & (eflux_rates < 0).any(axis=1)]))

non-zero 93
non-negative 75
non-positive 17
mixed sign 1


In [84]:
# Update the lower bound of reactions with non-negative flux
for x in eflux_rates.index[(eflux_rates >= 0).all(axis=1)]:
    r = reduced_model.reactions.get_by_id(x)
    r.lower_bound = 0.0

In [85]:
# Reverse reactions with non-positive flux
for x in eflux_rates.index[(eflux_rates <= 0).all(axis=1)]:
    r = reduced_model.reactions.get_by_id(x)
    r.id = r.id + 'r'
    r.lower_bound = 0.0
    for m, s in r.metabolites.items():
        r.add_metabolites({m: -2*s})
    eflux_rates.loc[x] = -eflux_rates.loc[x]
    eflux_rates.rename(index={x: x+'r'}, inplace=True)

In [86]:
# Split reactions with mixed signs
for x in eflux_rates.index[(eflux_rates > 0).any(axis=1) & (eflux_rates < 0).any(axis=1)]:
    r = reduced_model.reactions.get_by_id(x)
    r.lower_bound = 0.0
    r_reverse = r.copy()
    r_reverse.id = r.id+'r'
    reduced_model.add_reactions([r_reverse])
    for m, s in r_reverse.metabolites.items():
        r_reverse.add_metabolites({m: -2*s})
    eflux_rates.loc[x+'r'] = -eflux_rates.loc[x]
    eflux_rates.loc[x][eflux_rates.loc[x] < 0] = 0.0
    eflux_rates.loc[x+'r'][eflux_rates.loc[x+'r'] < 0] = 0.0

In [87]:
for r in reduced_model.reactions:
    if r.lower_bound < 0.0:
        print(r)

In [88]:
print('reduced', len(eflux_rates))
print('non-negative', len(eflux_rates[(eflux_rates >= 0).all(axis=1)]))
print('non-positive', len(eflux_rates[(eflux_rates <= 0).all(axis=1)]))
print('mixed sign', len(eflux_rates[(eflux_rates > 0).any(axis=1) & (eflux_rates < 0).any(axis=1)]))

reduced 94
non-negative 94
non-positive 0
mixed sign 0


In [89]:
# Check reactions with zero flux in the reference strain
eflux_rates[eflux_rates[ref_line] == 0]

,SF ABF180_1_R1,SF ABF180_1_R2,SF ABF180_1_R3,SF ABF180_2_R1,SF ABF180_2_R2,SF ABF180_2_R3,SF ABF180_3_R1,SF ABF180_3_R2,SF ABF180_3_R3,SF ABF180_4_R1,...,SF ABF180_21_R3,SF ABF180_22_R1,SF ABF180_22_R2,SF ABF180_22_R3,SF ABF180_23_R1,SF ABF180_23_R2,SF ABF180_23_R3,SF ABF180_24_R1,SF ABF180_24_R2,SF ABF180_24_R3
r28r,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.039755,0.005311


### Handle the reactions with zero flux in the reference strain

In [119]:
ref_zero_flux = eflux_rates.index[eflux_rates[ref_line] == 0]
eflux_rates.loc[ref_zero_flux][eflux_rates.loc[ref_zero_flux] > 0].min(axis=1)

r28r    0.005311
dtype: float64

In [117]:
# r28 and r28r can cancel out if adding the same value
eflux_rates[ref_line]['r28']

0.7959490545709038

In [116]:
# Calculate the maximum fold change in other reactions
temp = eflux_rates.divide(eflux_rates[ref_line], axis=0).max(axis=1)
temp[temp < np.inf].max()

5.313918048379364

In [120]:
# Adding 1e-3 to r28 and r28r in the reference strain, resulting in > 5 fold change
eflux_rates[ref_line]['r28'] = eflux_rates[ref_line]['r28'] + 0.001
eflux_rates[ref_line]['r28r'] = eflux_rates[ref_line]['r28r'] + 0.001

In [121]:
for x in ref_zero_flux:
    r = reduced_model.reactions.get_by_id(x)
    print(r)

r28r: F6P + T3P1 --> E4P + XUL5P


In [124]:
for k in eflux_rates:
    print(k)
    for m in reduced_model.metabolites:
        temp = 0
        for r in m.reactions:
            temp = temp + r.get_coefficient(m)*eflux_rates[k][r.id]
        if abs(temp) > 1e-9:
            print(m.id, temp)

SF ABF180_1_R1
SF ABF180_1_R2
SF ABF180_1_R3
SF ABF180_2_R1
SF ABF180_2_R2
SF ABF180_2_R3
SF ABF180_3_R1
SF ABF180_3_R2
SF ABF180_3_R3
SF ABF180_4_R1
SF ABF180_4_R2
SF ABF180_4_R3
SF ABF180_5_R1
SF ABF180_5_R2
SF ABF180_5_R3
SF ABF180_6_R1
SF ABF180_6_R2
SF ABF180_6_R3
SF ABF180_7_R1
SF ABF180_7_R2
SF ABF180_7_R3
SF ABF180_8_R1
SF ABF180_8_R2
SF ABF180_8_R3
SF ABF180_9_R1
SF ABF180_9_R2
SF ABF180_9_R3
SF ABF180_10_R1
SF ABF180_10_R2
SF ABF180_10_R3
SF ABF180_11_R1
SF ABF180_11_R2
SF ABF180_11_R3
SF ABF180_12_R1
SF ABF180_12_R2
SF ABF180_12_R3
SF ABF180_13_R1
SF ABF180_13_R2
SF ABF180_13_R3
SF ABF180_14_R1
SF ABF180_14_R2
SF ABF180_14_R3
SF ABF180_15_R1
SF ABF180_15_R2
SF ABF180_15_R3
SF ABF180_16_R1
SF ABF180_16_R2
SF ABF180_16_R3
SF ABF180_17_R1
SF ABF180_17_R2
SF ABF180_17_R3
SF ABF180_18_R1
SF ABF180_18_R2
SF ABF180_18_R3
SF ABF180_19_R1
SF ABF180_19_R2
SF ABF180_19_R3
SF ABF180_20_R1
SF ABF180_20_R2
SF ABF180_20_R3
SF ABF180_21_R1
SF ABF180_21_R2
SF ABF180_21_R3
SF ABF180_22_R1
SF 

## Save eflux rates and reduced metabolic model

In [125]:
eflux_rates.to_csv('../../data/round2/Eflux2_flux_rates_reduced_before_eflux.csv')

In [126]:
cobra.io.save_json_model(reduced_model, '../../models/iJB1325_HP_reduced_before_eflux_round2.json')